# Reading, validating and converting the HCMO ISA-Tab example


## Abstract

This notebook is adapted from the ISA-API tutorial *Reading ISA-Tab from files and validating ISA-Tab files*. It uses the hand-authored HCMO example in this directory (`isatab/`) and the HCMO ISA configuration in `config/` to:

- read and load an ISA-Tab investigation in memory;
- navigate key objects and pull key attributes;
- run the ISA-Tab validator with the HCMO XML configuration;
- convert the ISA-Tab to ISA-JSON with the HCMO JSON configuration and validate the result.

Run it from this directory (`examples/isa-authoring/`) with `isatools==0.14.3` installed (`pip install -r tooling/interoperability-requirements.txt` from the repository root). The same checks run headlessly in `tooling/validate_isa_authoring_example.py`.


## 1. Getting the tools

In [ ]:
# The pinned ISA-API version is listed in tooling/interoperability-requirements.txt.
# On Google Colab, uncomment and run:
# !pip install isatools==0.14.3


In [ ]:
import json
import os
import tempfile

from isatools import isajson, isatab, isatab2json

ISATAB_DIR = os.path.join("isatab")
INVESTIGATION = os.path.join(ISATAB_DIR, "i_Investigation_hcmo.txt")
XML_CONFIG_DIR = os.path.join("config", "xml")
JSON_CONFIG_DIR = os.path.join("config", "json", "default")


## 2. Reading and loading an ISA Investigation in memory from an ISA-Tab instance

In [ ]:
with open(INVESTIGATION) as fp:
    ISA = isatab.load(fp)


### Let's check the description of the first study object present in an ISA Investigation object

In [ ]:
ISA.studies[0].description

### Let's check the protocols declared in ISA the study (using a python list comprehension):

In [ ]:
[protocol.description for protocol in ISA.studies[0].protocols]

### Let's now checks the ISA Assay Measurement and Technology Types  are used in this ISA Study object

In [ ]:
[f'{assay.measurement_type.term} using {assay.technology_type.term}' for assay in ISA.studies[0].assays]

### Let's now check the `ISA Study Source` Material:

In [ ]:
[source.name for source in ISA.studies[0].sources]

#### Let's check what is the first `ISA Study Source property`:

In [ ]:
# here, we get all the characteristics of the first Source object
first_source_characteristics = ISA.studies[0].sources[0].characteristics

In [ ]:
first_source_characteristics[0].category.term

#### Let's now check what is the `value` associated with that first `ISA Study Source property`:

In [ ]:
first_source_characteristics[0].value

#### Let's now check what are all the properties associated with this first `ISA Study Source`

In [ ]:
[char.category.term for char in first_source_characteristics]

#### And the corresponding values are:

In [ ]:
[char.value for char in first_source_characteristics]

## 3. Invoking the python ISA-Tab validator with the HCMO XML configuration


In [ ]:
with open(INVESTIGATION) as fp:
    my_json_report_hcmo = isatab.validate(fp, config_dir=XML_CONFIG_DIR)


In [ ]:
my_json_report_hcmo

- This `Validation Report` shows that no error has been logged.
- The rest of the report consists in warnings meant to draw the attention of the curator to elements which may be provided but which do not break the ISA syntax.
- Notice the `study group` information reported on both study and assay files. If ISA `Factor Value[]` fields are found present in the `ISA Study` or `ISA Assay` tables, the validator will try to identify the set of unique `Factor Value` combinations defining a `Study Group`.
    - When no `Factor Value` are found in an ISA `Study` or `Assay` table, the value is left to its default value: -1, which means that no `Study Group` has been found.
    - ISA **strongly** encourages to declare Study Groups using ISA Factor Values to unambiguously identify the independent variables of an experiment.


## 4. Converting the ISA-Tab to ISA-JSON with the HCMO JSON configuration

`isatab2json.convert` takes the *directory* holding the ISA-Tab files and the directory holding the JSON configurations (`config/json/default`). Validation is run separately below, so `validate_first=False` here.


In [ ]:
hcmo_isa_json = isatab2json.convert(
    os.path.abspath(ISATAB_DIR),
    config_dir=os.path.abspath(JSON_CONFIG_DIR),
    use_new_parser=True,
    validate_first=False,
)
study = hcmo_isa_json["studies"][0]
print(study["identifier"], "|", len(study["materials"]["sources"]), "sources,", len(study["materials"]["samples"]), "samples")
print([assay["filename"] for assay in study["assays"]])


In [ ]:
# Validate the committed ISA-JSON (isajson/isa.json) against the HCMO JSON configuration.
with open(os.path.join("isajson", "isa.json")) as committed_fp:
    report = isajson.validate(committed_fp, config_dir=os.path.abspath(JSON_CONFIG_DIR))

print("errors:", len(report["errors"]), "| warnings:", len(report["warnings"]))
report


In [ ]:
# Write the freshly converted ISA-JSON to a temporary file (the committed copy is
# regenerated with `python tooling/validate_isa_authoring_example.py --write`).
with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as out_fp:
    json.dump(hcmo_isa_json, out_fp, indent=2)
print("written to", out_fp.name)


## About this notebook

- adapted from the ISA-API tutorial notebook by philippe.rocca-serra@oerc.ox.ac.uk (CC-BY 4.0), for the HCMO ISA authoring example
- license: CC-BY 4.0
- ISA-API support: isatools@googlegroups.com, issue tracker: https://github.com/ISA-tools/isa-api/issues
